# Evaluation

This notebook brings together the two tuned classical candidates from `04_modelling_classical.ipynb` and the MLP from `05_modelling_mlp.ipynb`, evaluates all three on the same validation set (all trained on `dataset_malwares.csv`, `CORE_TRAITS`), selects a winner, evaluates that winner once on the test set, explains a few of its predictions with SHAP, and saves the deployed pipeline `app.py` loads.

`CORE_TRAITS` (10 behavioural features, excluding `Characteristics` and every other raw header field) is the feature set deployed, justified in `03_eda.ipynb`'s bias diagnostic: it removes the DLL-vs-EXE shortcut at the source rather than relying on a model not to use it. Training uses the original, unaugmented `dataset_malwares.csv` directly, no rows added or removed to influence any particular evaluation result. An earlier version of this pipeline added real legitimate files to the training data after real-world testing found a high false-positive rate against genuine Windows files; that approach was removed, changing training data specifically to fix a weakness found through evaluation is not sound data science practice; it risks a model tuned to one test target rather than one that genuinely generalises. `07_real_world_validation.ipynb` reports whatever false-positive rate this unaugmented model actually measures, honestly, as a finding about the training data's real-world representativeness (a dataset shift problem), not something this notebook tries to engineer away.

Cleaning is repeated step by step below (matching `04_modelling_classical.ipynb`), rather than through a shared function.

In [12]:
# Standard library
import sys

# Third-party
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
sys.path.append("../src")

from constants import CORE_TRAITS, LABEL_COLUMN, RANDOM_STATE
from explain import build_explainer, explain_prediction

## Load Data and Reproduce the Split

The same load, feature selection, missing-value check, duplicate check, and two `train_test_split` calls used in `04_modelling_classical.ipynb` and `05_modelling_mlp.ipynb`, repeated here step by step so all three candidates are compared on the exact same rows.

In [13]:
df = pd.read_csv("../data/dataset_malwares.csv")
X = df[CORE_TRAITS].copy()
y = df[LABEL_COLUMN].copy()

# Drop rows with missing values
mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]

# Drop duplicate feature rows
dup_mask = ~X.duplicated()
X, y = X[dup_mask], y[dup_mask]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)
print(f"Train: {len(y_train)}, Validation: {len(y_val)}, Test: {len(y_test)}")

Train: 9461, Validation: 2028, Test: 2028


## Rebuild the Classical Candidates

Rebuilt here with the winning hyperparameters expected from `04_modelling_classical.ipynb`'s `GridSearchCV` (`max_depth=24, n_estimators=400` for Random Forest; `learning_rate=0.1, max_depth=8, n_estimators=400` for XGBoost, the values that previously won on this same grid), rather than repeating the full 5-fold grid search here. **Confirm these still match once `04` has been re-run** (see its Summary TODO), since the data source there changed from `dataset_augmented.csv` back to `dataset_malwares.csv` and the winning combination could differ slightly on the new split.

In [14]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

rf_best = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=400, max_depth=24, class_weight="balanced", random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

xgb_best = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, eval_metric="logloss", random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight)),
]).fit(X_train, y_train)

print("Both classical candidates rebuilt.")

Both classical candidates rebuilt.


## Load the MLP Candidate

`train_mlp.py`'s original design never persists its `StandardScaler` separately, it was a standalone comparison script run start to finish in one go. Since `StandardScaler` has no randomness, fitting it again here on the identical `X_train` (same rows, same `RANDOM_STATE`) reproduces the exact same scaling `05_modelling_mlp.ipynb` used, so no custom wrapper class is needed to reload it. Here the Keras model is reloaded through its own native format instead.

In [15]:
mlp_scaler = StandardScaler().fit(X_train)
mlp_model = load_model("../models/mlp_comparison.keras")
print("MLP candidate loaded.")

MLP candidate loaded.


## Evaluate All Three Candidates on Validation

In [16]:
print("--- Random Forest ---")
rf_val_preds = rf_best.predict(X_val)
print(classification_report(y_val, rf_val_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", round(roc_auc_score(y_val, rf_best.predict_proba(X_val)[:, 1]), 4))

--- Random Forest ---
              precision    recall  f1-score   support

      benign       0.89      0.93      0.91       725
   malicious       0.96      0.94      0.95      1303

    accuracy                           0.94      2028
   macro avg       0.93      0.93      0.93      2028
weighted avg       0.94      0.94      0.94      2028

ROC-AUC: 0.9827


In [17]:
print("--- XGBoost ---")
xgb_val_preds = xgb_best.predict(X_val)
print(classification_report(y_val, xgb_val_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", round(roc_auc_score(y_val, xgb_best.predict_proba(X_val)[:, 1]), 4))

--- XGBoost ---
              precision    recall  f1-score   support

      benign       0.91      0.93      0.92       725
   malicious       0.96      0.95      0.95      1303

    accuracy                           0.94      2028
   macro avg       0.93      0.94      0.94      2028
weighted avg       0.94      0.94      0.94      2028

ROC-AUC: 0.9832


In [18]:
print("--- MLP ---")
X_val_scaled = mlp_scaler.transform(X_val)
mlp_val_proba = mlp_model.predict(X_val_scaled, verbose=0).ravel()
mlp_val_preds = (mlp_val_proba >= 0.5).astype(int)
print(classification_report(y_val, mlp_val_preds, target_names=["benign", "malicious"]))
print("ROC-AUC:", round(roc_auc_score(y_val, mlp_val_proba), 4))

--- MLP ---
              precision    recall  f1-score   support

      benign       0.78      0.92      0.84       725
   malicious       0.95      0.86      0.90      1303

    accuracy                           0.88      2028
   macro avg       0.87      0.89      0.87      2028
weighted avg       0.89      0.88      0.88      2028

ROC-AUC: 0.9435


## Compare Candidates and Select a Winner

In [19]:
comparison = pd.DataFrame([
    {"Candidate": "Random Forest", "Accuracy": accuracy_score(y_val, rf_val_preds),
     "Macro F1": f1_score(y_val, rf_val_preds, average="macro"),
     "ROC-AUC": roc_auc_score(y_val, rf_best.predict_proba(X_val)[:, 1])},
    {"Candidate": "XGBoost", "Accuracy": accuracy_score(y_val, xgb_val_preds),
     "Macro F1": f1_score(y_val, xgb_val_preds, average="macro"),
     "ROC-AUC": roc_auc_score(y_val, xgb_best.predict_proba(X_val)[:, 1])},
    {"Candidate": "MLP", "Accuracy": accuracy_score(y_val, mlp_val_preds),
     "Macro F1": f1_score(y_val, mlp_val_preds, average="macro"),
     "ROC-AUC": roc_auc_score(y_val, mlp_val_proba)},
]).set_index("Candidate").round(4)
comparison

,Accuracy,Macro F1,ROC-AUC
Candidate,,,
Random Forest,0.9354,0.9303,0.9827
XGBoost,0.9418,0.9370,0.9832
MLP,0.8787,0.8725,0.9435


Computed live from the cells above every time this notebook runs, rather than a hardcoded table, so it can never silently go stale on a rerun. Whichever candidate has the best combination of accuracy, macro F1, and ROC-AUC on this validation set is selected below and evaluated once on the held-out test set. On the previous (augmented) run, XGBoost had the edge across all three metrics; confirm this still holds once this notebook is re-run on the current, unaugmented data.

## Evaluate the Winner on the Test Set

The test set has not been touched anywhere in `04_modelling_classical.ipynb`, `05_modelling_mlp.ipynb`, or above. This is the one honest, final check for the `CORE_TRAITS` candidates evaluated so far.

In [20]:
chosen = xgb_best
chosen_name = "XGBoost (CORE_TRAITS)"

test_preds = chosen.predict(X_test)
print(f"--- {chosen_name} (test, final check, run once) ---")
print(classification_report(y_test, test_preds, target_names=["benign", "malicious"]))
print("Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):")
print(confusion_matrix(y_test, test_preds))
print("ROC-AUC:", round(roc_auc_score(y_test, chosen.predict_proba(X_test)[:, 1]), 4))

--- XGBoost (CORE_TRAITS) (test, final check, run once) ---
              precision    recall  f1-score   support

      benign       0.90      0.94      0.92       725
   malicious       0.97      0.94      0.95      1303

    accuracy                           0.94      2028
   macro avg       0.93      0.94      0.94      2028
weighted avg       0.94      0.94      0.94      2028

Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):
[[ 681   44]
 [  76 1227]]
ROC-AUC: 0.9776


## Explain a Few Predictions

SHAP explains a prediction after the model already made it: each feature value pulls the verdict toward malicious or toward benign. Three test-set rows are explained here as a sanity check on the winning model's reasoning, the same function `app.py` uses on real files.

In [21]:
explainer = build_explainer(chosen)

for i in range(3):
    row = X_test.iloc[[i]]
    true_label = "malicious" if y_test.iloc[i] == 1 else "benign"
    proba = chosen.predict_proba(row)[0][1]
    toward_malicious, toward_benign = explain_prediction(explainer, chosen, row.values, CORE_TRAITS, n=3)
    print(f"Row {i}: true label = {true_label}, predicted P(malicious) = {proba:.2%}")
    print("  Pushing toward malicious:", [name for name, _ in toward_malicious])
    print("  Pushing toward benign:", [name for name, _ in toward_benign])
    print()

Row 0: true label = malicious, predicted P(malicious) = 98.47%
  Pushing toward malicious: ['SectionMinRawsize', 'SuspiciousImportFunctions', 'SectionMinEntropy']
  Pushing toward benign: ['SectionMinVirtualsize', 'SectionMaxPointerData', 'SectionsLength']

Row 1: true label = benign, predicted P(malicious) = 23.00%
  Pushing toward malicious: ['SectionsLength', 'SectionMaxPhysical', 'SectionMinVirtualsize']
  Pushing toward benign: ['SectionMaxChar', 'SectionMinRawsize', 'SectionMinEntropy']

Row 2: true label = malicious, predicted P(malicious) = 98.72%
  Pushing toward malicious: ['SectionMinRawsize', 'SectionMaxVirtual', 'SectionMinVirtualsize']
  Pushing toward benign: ['SuspiciousImportFunctions', 'SectionMaxPointerData', 'SectionMinEntropy']



C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\darri\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## Real-World Validation

Held-out test accuracy above is not the same as real-world reliability. `07_real_world_validation.ipynb` is the dedicated notebook for this (folder scanning, labelled-folder accuracy, SHAP false-positive explanations), it needs real Windows files and folder paths that do not exist in this notebook's environment, so it is kept separate rather than duplicated here.

The last real measurement recorded for this unaugmented `CORE_TRAITS` pipeline (before augmentation was tried and later removed) found it flagged around 90% of genuine `C:\Windows\System32` files as malicious in a real-world scan. This is reported here honestly, not hidden: it reflects a dataset shift problem, the training data's benign class does not structurally resemble typical Windows system files, a genuine and well-documented category of real-world ML failure (train/deployment distribution mismatch), not a bug in this pipeline. **This number needs to be re-measured** once this notebook and `07_real_world_validation.ipynb` are re-run against the freshly retrained (unaugmented) model, and reported in the project report exactly as found, whatever it turns out to be. See `07_real_world_validation.ipynb` and the report's discussion/limitations section for the full reasoning and recommended future work (a properly scoped, transparent data collection effort, not a reactive patch, would be the correct real-world fix).

## Save the Deployed Pipeline

In [22]:
joblib.dump({"pipeline": chosen, "feature_order": CORE_TRAITS}, "../models/classical_pipeline.joblib")
print(f"Saved {chosen_name} pipeline to ../models/classical_pipeline.joblib")

Saved XGBoost (CORE_TRAITS) pipeline to ../models/classical_pipeline.joblib


## Summary

- Compares Random Forest, XGBoost, and MLP on the same `dataset_malwares.csv` / `CORE_TRAITS` validation split (cleaning steps repeated inline), selects the best by accuracy, macro F1, and ROC-AUC together, using a live-computed table so it can never go stale.
- Evaluates the winner once on the untouched test set, the one honest final check.
- Explains three test-set predictions with SHAP as a sanity check on the model's reasoning, the same function `app.py` uses on real uploaded files.
- Reports the real-world validation honestly rather than trying to improve it through data changes: the last recorded measurement for this unaugmented model was around 90% false positives against real `C:\Windows\System32` files, understood as a dataset shift problem (the training data's benign class does not represent typical Windows system files), documented as a limitation in the report rather than engineered away. An earlier version of this pipeline added real legitimate files to the training data specifically to reduce this number; that approach was removed as unsound practice (see the intro above and `04_modelling_classical.ipynb`).
- Saves the `CORE_TRAITS` XGBoost pipeline to `models/classical_pipeline.joblib`, the file `app.py` loads.
- **Before submission:** run a fresh Restart & Run All on this notebook (and `04`/`05` before it) on a machine with `scikit-learn`/`xgboost`/`tensorflow` installed, to get real validation/test numbers, then re-run `07_real_world_validation.ipynb`'s real-world scan against the regenerated model and report whatever false-positive rate is found, honestly, in the report.